In [105]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.edge.service import Service
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver import ActionChains
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import NoSuchElementException, StaleElementReferenceException

In [79]:
def strip_parent(string):
    return string.split('(')[1]

def switch_date(driver, go_to_date):
    '''
    go_to_date[int]: 要切换到的日期
    '''
    date_xpath = '/html/body/div[1]/div/main/div/div[2]/div[1]/div[1]/div[1]/div[2]/div/div/div/div[2]/button[{num_date}]/div/span'.format(num_date=str(go_to_date))
    date_element = driver.find_element(By.XPATH, date_xpath)
    date = date_element.text
    date_element.click()
    print('已切换至',date,'日')
    
def get_match(driver):
    '''
    获取完场比赛比分及盘口信息
    '''
    #获取比分
    home_score = driver.find_element(By.XPATH,'/html/body/div[1]/div/main/div/div[2]/div[3]/div/div[1]/div/div[1]/div[2]/div[2]/div[2]/div/div[1]/div[1]/span').text
    away_score = driver.find_element(By.XPATH,'/html/body/div[1]/div/main/div/div[2]/div[3]/div/div[1]/div/div[1]/div[2]/div[2]/div[2]/div/div[1]/div[3]/span').text

    #打开新标签页获取盘口
    element = driver.find_element(By.XPATH,'/html/body/div[1]/div/main/div/div[2]/div[3]/div/div[1]/div/a/button') #Show More Button
    element.send_keys(Keys.CONTROL + Keys.RETURN)
    driver.switch_to.window(driver.window_handles[1])
    
    #存储当前比赛数据                      
    handicap_H = driver.find_element(By.XPATH,'/html/body/div[1]/div/main/div[2]/div[2]/div[1]/div[1]/div[2]/div[3]/div[2]/div/div[1]/span').text #主盘口
    handicap_A = driver.find_element(By.XPATH,'/html/body/div[1]/div/main/div[2]/div[2]/div[1]/div[1]/div[2]/div[3]/div[2]/div/div[2]/span').text #客盘口
    value_home = driver.find_element(By.XPATH,'/html/body/div[1]/div/main/div[2]/div[2]/div[1]/div[1]/div[2]/div[3]/div[2]/div/div[1]/div/span').text #主赔
    value_away = driver.find_element(By.XPATH,'/html/body/div[1]/div/main/div[2]/div[2]/div[1]/div[1]/div[2]/div[3]/div[2]/div/div[2]/div/span').text #客赔

    #回到主页面
    driver.close()
    driver.switch_to.window(driver.window_handles[0])
    return [home_score, away_score, handicap_H ,handicap_A ,value_home ,value_away]

In [32]:
#driver_path = r'D:\edgedriver_win64\msedgedriver.exe' #PC
driver_path = r'D:\EdgeDriver\msedgedriver.exe' #company
driver = webdriver.Edge(service=Service(executable_path=driver_path))
driver.implicitly_wait(10)
driver.get("https://www.sofascore.com/")

#显示赔率
showodds = driver.find_element(By.CLASS_NAME,'slider')
showodds.click()
print('初始化完成！')

In [63]:
switch_date(driver, 28) #切换日期

已切换至 28 日


In [70]:
info_list = get_match(driver)

In [71]:
info_list

['1', '0', '(-2) Atlético Mineiro', '(2) Juventude', '2.05', '1.80']

In [92]:
#划动比赛
ActionChains(driver).scroll_by_amount(0, 950).perform()

In [108]:
#数据存储
#FIXME 1.解决划动问题 2.如何根据数据库缺盘口的比赛寻找sofascore比赛
#matches = driver.find_elements(By.CLASS_NAME,'sc-hLBbgP.dRtNhU.sc-9199a964-1.kusmLq')
for i in range(2,18):
    match_xpath = '/html/body/div[1]/div/main/div/div[2]/div[2]/div/div[2]/div/div/div[{index}]/a/div/div/div[1]'.format(index=str(i))
    try:
        match = driver.find_element(By.XPATH, match_xpath)
        match.click()
        #判断比赛状况
        try:
            #未完赛比赛                                 
            handicap_H = driver.find_element(By.XPATH,'/html/body/div[1]/div/main/div/div[2]/div[3]/div/div[1]/div/div[4]/div/div/div[3]/div[3]/div[2]/div/div[1]/span')
            handicap_A = driver.find_element(By.XPATH,'/html/body/div[1]/div/main/div/div[2]/div[3]/div/div[1]/div/div[4]/div/div/div[3]/div[3]/div[2]/div/div[2]/span')
            value_home = driver.find_element(By.XPATH,'/html/body/div[1]/div/main/div/div[2]/div[3]/div/div[1]/div/div[4]/div/div/div[3]/div[3]/div[2]/div/div[1]/div/span')
            value_away = driver.find_element(By.XPATH,'/html/body/div[1]/div/main/div/div[2]/div[3]/div/div[1]/div/div[4]/div/div/div[3]/div[3]/div[2]/div/div[2]/div/span')
        except:
            pass
    
        try:
            #完赛比赛
            info_list = get_match(driver)
            home_score = info_list[0]
            away_score = info_list[1]
            handicap_H = info_list[2]
            handicap_A = info_list[3]
            value_home = info_list[4]
            value_away = info_list[5]
        except:
            pass

        try:
            home_name = handicap_H.split(') ')[1]
            away_name = handicap_A.split(') ')[1]
            print(home_name+' - '+away_name)
            print('盘口：', strip_parent(handicap_H.split(') ')[0]))
        except:
            pass
    except NoSuchElementException:
        pass

FC Zürich - Bodø/Glimt
盘口： 0.25
PSV Eindhoven - Arsenal
盘口： 0.25
PSV Eindhoven - Arsenal
盘口： 0.25
Fenerbahçe - Stade Rennais
盘口： 0
Ludogorets Razgrad - Real Betis Balompié
盘口： 0
HJK - Roma
盘口： 1.5
1. FC Union Berlin - Sporting Braga
盘口： -0.5
Malmö FF - Royale Union Saint-Gilloise
盘口： 0.75
